In [10]:
import sys
from pathlib import Path
import torch

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
import Python.simfun as sim


X1, y1, beta1, sim_info1 = sim.simfun1(
    sim="simple",
    n=160,
    p=100,
    seed=400,
    n_active=10,
    sigma2=1,
    device=device,
    dtype=torch.float32,
)


X2, y2, beta2, sim_info2 = sim.simfun1(
    sim="simple",
    n=100,
    p=500,
    seed=400,
    n_active=10,
    sigma2=1,
    device=device,
    dtype=torch.float32,
)


In [ ]:
%load_ext autoreload
%autoreload 2

import Python.config as cfg
import Python.framework as fw
import Python.model as md


split_cfg = cfg.SplitConfig(
    train_frac=0.6,
    val_frac=0.2,
    test_frac=0.2,
    seed=123,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
schedule_cfg_test = cfg.StagewiseAnnealConfig(
    tau_start=0.8,
    tau_end=0.8,
    warmup_epochs=100,
    n_anneal_stages=1,
    min_stage_epochs=1000,
    max_stage_epochs=1000,
    base_lr=1e-5,
    stage_lr_decay=1,
    min_lr_scale=1,
    eval_every=25,
    print_every=100,
    diag_R_train=64,
    diag_R_final=4000,
    support_threshold=0.5,
    q_entropy_weight=0.0,
    recovery_min_epoch=25,
    recovery_score_col="moment_recovery_score",
)

out_lasso_test = fw.simflow_stagewise(
    X=X1,
    y=y1,
    beta_true=beta1,
    sim_info=sim_info1,
    build_flow_vi=md.build_flow_vi,
    family="gaussian",
    seed=400,
    K_q=64,
    K_g=4,
    schedule_cfg=schedule_cfg_test,
    split_cfg=split_cfg,
    save_cfg=cfg.SaveConfig(output_dir=None),
    mode="lasso_recovery",
)

===== Simulation info =====
Simulation summary:
  sim                   : simple
  n                     : 160
  p                     : 100
  n_active              : 10
  sigma2                : 1.0000
  sigma                 : 1.0000
  signal_var            : 20.4217
  outcome_var           : 20.6988
  snr_actual            : 20.4217
  beta_low              : 0.3000
  beta_high             : 2.0000
  center_y              : True

Active indices, zero-based:
  [12, 18, 25, 31, 38, 49, 52, 81, 89, 97]

Active variables sorted by |beta_true|:
  rank     j    j1    beta_true       |beta|
     1    81    82       1.9855       1.9855
     2    38    39      -1.8608       1.8608
     3    49    50       1.7504       1.7504
     4    31    32       1.7441       1.7441
     5    97    98      -1.7155       1.7155
     6    18    19      -1.6516       1.6516
     7    12    13       1.2005       1.2005
     8    52    53      -1.1844       1.1844
     9    25    26      -1.1523       1.1523
  